# PIRLS Maroc — prétraitement robuste et comparaison des imputations

Ce notebook est une nouvelle version indépendante de `Main_Revision.ipynb`. Il ne modifie ni l'ancien notebook ni ses résultats.

Objectifs :
- auditer les codes et taux de valeurs manquantes ;
- exclure les valeurs plausibles de lecture du clustering ;
- supprimer les colonnes vides, constantes ou trop incomplètes ;
- appliquer médiane aux variables continues et mode aux variables discrètes ;
- réaliser une sensibilité par imputation itérative de type MICE ;
- recalculer les méthodes classiques et profondes ;
- comparer les partitions avec l'Adjusted Rand Index (ARI).

**Important :** la liste `MISSING_CODES` doit être vérifiée avec les dictionnaires PIRLS correspondant exactement aux fichiers utilisés. Les codes génériques 9/99 ne sont volontairement pas traités automatiquement, car ils peuvent être des valeurs valides selon la variable.


In [ ]:
# 1. Imports et configuration
from pathlib import Path
import copy
import random
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_rand_score, silhouette_score,
    calinski_harabasz_score, davies_bouldin_score
)

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch absent : les méthodes profondes seront ignorées.")

# Le notebook fonctionne depuis la racine du projet ou depuis Data/.
HERE = Path.cwd()
DATA_DIR = HERE if (HERE / "ASPMARR5.csv").exists() else HERE / "Data"
if not (DATA_DIR / "ASPMARR5.csv").exists():
    raise FileNotFoundError("ASPMARR5.csv introuvable. Ouvrez le projet à sa racine ou exécutez depuis Data/.")

OUT_DIR = DATA_DIR / "Robust_imputation_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_CLUSTERS = 5
HIGH_MISSING_THRESHOLD = 0.50
LOW_CARDINALITY_MAX = 10
N_MICE_DRAWS = 5
MICE_MAX_ITER = 10
MICE_NEAREST_FEATURES = 30

# Valeurs déjà traitées comme manquantes dans le code historique.
# À confirmer variable par variable à l'aide des dictionnaires PIRLS.
MISSING_CODES = [-99, -98, -97, -96, -95, 999, 9999]

# Même nombre maximal de variables par source que dans Main_Revision.ipynb.
STU_MAX, TCH_MAX, SCH_MAX = 200, 120, 120

# True = exécuter AE+KMeans, DEC et IDEC pour le scénario principal et MICE-1.
RUN_DEEP_MODELS = True
PRETRAIN_EPOCHS = 100
DEEP_MAX_ITER = 2000
BATCH_SIZE = 256
LATENT_DIM = 16
HIDDEN_DIMS = [256, 64]
LEARNING_RATE = 1e-3
IDEC_RECON_WEIGHT = 1.0

# None calcule la silhouette sur tout l'échantillon. Mettre 3000 pour accélérer un essai.
SILHOUETTE_SAMPLE_SIZE = None

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

set_seed()
DEVICE = torch.device("cuda" if TORCH_AVAILABLE and torch.cuda.is_available() else "cpu") if TORCH_AVAILABLE else None
print("DATA_DIR =", DATA_DIR.resolve())
print("OUT_DIR  =", OUT_DIR.resolve())
print("DEVICE   =", DEVICE)


## 2. Chargement, codes manquants et exclusion des valeurs plausibles

Les cinq valeurs plausibles `ASRREA01`–`ASRREA05` ne doivent jamais entrer dans la matrice de clustering. L'assertion placée plus bas arrête le notebook si une valeur plausible est détectée.


In [ ]:
# 2. Chargement et fonctions de préparation
stu_raw = pd.read_csv(DATA_DIR / "ASPMARR5.csv", low_memory=False)
tch_raw = pd.read_csv(DATA_DIR / "ATGMARR5.csv", low_memory=False)
sch_raw = pd.read_csv(DATA_DIR / "ACGMARR5.csv", low_memory=False)

def replace_missing_codes(df):
    out = df.copy()
    feature_cols = [c for c in out.columns if c not in {"IDCNTRY", "IDSCHOOL", "IDSTUD", "IDTEACH"}]
    out[feature_cols] = out[feature_cols].replace(MISSING_CODES, np.nan)
    return out

def is_plausible_value(name):
    # PIRLS overall reading and common achievement-PV naming families.
    u = str(name).upper()
    return bool(re.match(r"^(ASRREA|ASRLIT|ASRINF)0[1-5]$", u)) or "PLAUSIBLE" in u

stu = replace_missing_codes(stu_raw)
tch = replace_missing_codes(tch_raw)
sch = replace_missing_codes(sch_raw)

pv_found_in_sources = {
    "student": [c for c in stu.columns if is_plausible_value(c)],
    "teacher": [c for c in tch.columns if is_plausible_value(c)],
    "school": [c for c in sch.columns if is_plausible_value(c)],
}
print("Valeurs plausibles trouvées dans les trois CSV :", pv_found_in_sources)

def numeric_candidates(df, exclude, prefix, cap, robust):
    cols = [
        c for c in df.select_dtypes(include=[np.number]).columns
        if c not in exclude and not is_plausible_value(c)
    ]
    audit = pd.DataFrame({
        "source_column": cols,
        "missing_rate": [df[c].isna().mean() for c in cols],
        "n_unique_observed": [df[c].nunique(dropna=True) for c in cols],
    })
    if robust:
        audit["kept_before_cap"] = (
            (audit["missing_rate"] <= HIGH_MISSING_THRESHOLD) &
            (audit["n_unique_observed"] > 1)
        )
        cols = audit.loc[audit["kept_before_cap"], "source_column"].tolist()[:cap]
    else:
        audit["kept_before_cap"] = True
        cols = cols[:cap]
    block = df[cols].copy()
    block.columns = [prefix + c for c in cols]
    audit["source"] = prefix.rstrip("_")
    audit["scenario"] = "robust" if robust else "legacy"
    audit["selected_after_cap"] = audit["source_column"].isin(cols)
    return block, audit

def infer_block_discrete(block):
    discrete = []
    for c in block.columns:
        observed = block[c].dropna().to_numpy(dtype=float)
        if len(observed) == 0:
            continue
        integer_like = np.all(np.isclose(observed, np.round(observed)))
        if integer_like and pd.Series(observed).nunique() <= LOW_CARDINALITY_MAX:
            discrete.append(c)
    return discrete

def first_mode(series):
    modes = series.mode(dropna=True)
    return modes.iloc[0] if len(modes) else np.nan

def aggregate_at_school(part, feature_cols, discrete_cols, robust):
    if not robust:
        return part.groupby(["IDCNTRY", "IDSCHOOL"], as_index=False).mean(numeric_only=True)
    agg_rules = {c: (first_mode if c in discrete_cols else "mean") for c in feature_cols}
    return part.groupby(["IDCNTRY", "IDSCHOOL"], as_index=False).agg(agg_rules)

def build_merged_matrix(robust):
    stu_blk, a_stu = numeric_candidates(stu, {"IDCNTRY", "IDSCHOOL", "IDSTUD"}, "STU_", STU_MAX, robust)
    tch_blk, a_tch = numeric_candidates(tch, {"IDCNTRY", "IDSCHOOL", "IDTEACH"}, "TCH_", TCH_MAX, robust)
    sch_blk, a_sch = numeric_candidates(sch, {"IDCNTRY", "IDSCHOOL"}, "SCH_", SCH_MAX, robust)

    stu_part = pd.concat([stu[["IDCNTRY", "IDSCHOOL", "IDSTUD"]].reset_index(drop=True), stu_blk.reset_index(drop=True)], axis=1)
    tch_part = pd.concat([tch[["IDCNTRY", "IDSCHOOL"]].reset_index(drop=True), tch_blk.reset_index(drop=True)], axis=1)
    sch_part = pd.concat([sch[["IDCNTRY", "IDSCHOOL"]].reset_index(drop=True), sch_blk.reset_index(drop=True)], axis=1)

    stu_discrete = infer_block_discrete(stu_blk) if robust else []
    tch_discrete = infer_block_discrete(tch_blk) if robust else []
    sch_discrete = infer_block_discrete(sch_blk) if robust else []
    tch_school = aggregate_at_school(tch_part, list(tch_blk.columns), tch_discrete, robust)
    sch_school = aggregate_at_school(sch_part, list(sch_blk.columns), sch_discrete, robust)
    merged = stu_part.merge(tch_school, on=["IDCNTRY", "IDSCHOOL"], how="left")
    merged = merged.merge(sch_school, on=["IDCNTRY", "IDSCHOOL"], how="left")
    ids = merged[["IDSTUD", "IDCNTRY", "IDSCHOOL"]].copy()
    Xdf = merged.drop(columns=["IDSTUD", "IDCNTRY", "IDSCHOOL"])
    assert not any(is_plausible_value(c.replace("STU_", "").replace("TCH_", "").replace("SCH_", "")) for c in Xdf.columns), "Une valeur plausible est entrée dans X."
    return ids, Xdf, pd.concat([a_stu, a_tch, a_sch], ignore_index=True), (stu_discrete + tch_discrete + sch_discrete)

ids_legacy, X_legacy_raw, audit_legacy, _ = build_merged_matrix(robust=False)
ids, X_robust_raw, audit_robust, robust_discrete_source = build_merged_matrix(robust=True)
assert ids["IDSTUD"].equals(ids_legacy["IDSTUD"])

source_audit = pd.concat([audit_legacy, audit_robust], ignore_index=True)
source_audit.to_csv(OUT_DIR / "source_variable_audit.csv", index=False)

print("Matrice historique avant imputation :", X_legacy_raw.shape)
print("Matrice robuste avant imputation    :", X_robust_raw.shape)
print("Colonnes totalement vides (historique) :", int(X_legacy_raw.isna().all().sum()))
print("Valeurs plausibles dans X robuste :", [c for c in X_robust_raw.columns if is_plausible_value(c)])


## 3. Audit final et trois scénarios d'imputation

- `legacy_median` reproduit autant que possible l'ancien traitement : médiane pour toutes les colonnes numériques disponibles.
- `robust_median_mode` est le scénario principal : filtrage, médiane pour les variables continues et mode pour les variables discrètes.
- `mice_1` à `mice_5` sont des imputations itératives stochastiques. Les catégories imputées sont ramenées à une modalité réellement observée.


In [ ]:
# 3. Audit final et imputation
def final_filter(Xdf, threshold=None):
    info = pd.DataFrame({
        "variable": Xdf.columns,
        "missing_rate": Xdf.isna().mean().values,
        "n_unique_observed": Xdf.nunique(dropna=True).values,
    })
    keep = info["n_unique_observed"].gt(1)
    if threshold is not None:
        keep &= info["missing_rate"].le(threshold)
    kept = info.loc[keep, "variable"].tolist()
    info["kept"] = info["variable"].isin(kept)
    return Xdf[kept].copy(), info

# Historique : seules les colonnes impossibles à imputer (vides/constantes) sont retirées explicitement.
X_legacy_kept, audit_final_legacy = final_filter(X_legacy_raw, threshold=None)
# Principal : seuil explicite de 50 %, puis retrait des constantes.
X_robust_kept, audit_final_robust = final_filter(X_robust_raw, threshold=HIGH_MISSING_THRESHOLD)

# Les types ont été déterminés avant l'agrégation. Les catégories enseignant/école
# ont été agrégées par mode, tandis que les variables continues ont été moyennées.
discrete_cols = [c for c in robust_discrete_source if c in X_robust_kept.columns]
continuous_cols = [c for c in X_robust_kept.columns if c not in discrete_cols]

def scale_frame(df):
    scaler = StandardScaler()
    arr = scaler.fit_transform(df).astype(np.float32)
    return arr, scaler

# Scénario historique
legacy_imp = SimpleImputer(strategy="median")
legacy_df = pd.DataFrame(legacy_imp.fit_transform(X_legacy_kept), columns=X_legacy_kept.columns, index=X_legacy_kept.index)
X_legacy, scaler_legacy = scale_frame(legacy_df)

# Scénario principal médiane/mode
robust_df = X_robust_kept.copy()
if continuous_cols:
    robust_df[continuous_cols] = SimpleImputer(strategy="median").fit_transform(robust_df[continuous_cols])
if discrete_cols:
    robust_df[discrete_cols] = SimpleImputer(strategy="most_frequent").fit_transform(robust_df[discrete_cols])
X_robust, scaler_robust = scale_frame(robust_df)

# Imputations itératives stochastiques de type MICE pour l'analyse de sensibilité.
mice_arrays = {}
mice_unscaled = {}
observed_categories = {c: np.sort(X_robust_kept[c].dropna().unique()) for c in discrete_cols}

for draw in range(1, N_MICE_DRAWS + 1):
    imp = IterativeImputer(
        random_state=SEED + draw,
        max_iter=MICE_MAX_ITER,
        sample_posterior=True,
        initial_strategy="median",
        n_nearest_features=min(MICE_NEAREST_FEATURES, X_robust_kept.shape[1]),
        skip_complete=True,
    )
    completed = pd.DataFrame(imp.fit_transform(X_robust_kept), columns=X_robust_kept.columns, index=X_robust_kept.index)
    # Respecter le domaine des variables discrètes : modalité observée la plus proche.
    for c, allowed in observed_categories.items():
        values = completed[c].to_numpy()
        completed[c] = allowed[np.abs(values[:, None] - allowed[None, :]).argmin(axis=1)]
    mice_unscaled[draw] = completed
    mice_arrays[draw], _ = scale_frame(completed)
    print(f"Imputation MICE {draw}/{N_MICE_DRAWS} terminée")

audit_final_legacy["scenario"] = "legacy"
audit_final_robust["scenario"] = "robust"
pd.concat([audit_final_legacy, audit_final_robust], ignore_index=True).to_csv(OUT_DIR / "final_missingness_audit.csv", index=False)
pd.DataFrame({
    "variable": X_robust_kept.columns,
    "inferred_type": ["discrete_mode" if c in discrete_cols else "continuous_median" for c in X_robust_kept.columns],
    "missing_rate": X_robust_kept.isna().mean().values,
}).to_csv(OUT_DIR / "robust_variable_types.csv", index=False)

preprocessing_summary = pd.DataFrame([
    {"scenario": "legacy_median", "n_rows": len(X_legacy), "n_features": X_legacy.shape[1], "imputation": "median_all", "missing_threshold": "none"},
    {"scenario": "robust_median_mode", "n_rows": len(X_robust), "n_features": X_robust.shape[1], "imputation": "median_continuous_mode_discrete", "missing_threshold": HIGH_MISSING_THRESHOLD},
    {"scenario": "mice_sensitivity", "n_rows": len(X_robust), "n_features": X_robust.shape[1], "imputation": f"iterative_stochastic_{N_MICE_DRAWS}_draws", "missing_threshold": HIGH_MISSING_THRESHOLD},
])
preprocessing_summary.to_csv(OUT_DIR / "preprocessing_summary.csv", index=False)
display(preprocessing_summary)
print("Variables discrètes (mode) :", len(discrete_cols))
print("Variables continues (médiane) :", len(continuous_cols))


## 4. Méthodes classiques et comparaison par ARI

Les numéros de clusters sont arbitraires. L'ARI compare les partitions sans nécessiter de renuméroter les clusters.


In [ ]:
# 4. Méthodes classiques
def clustering_metrics(X_space, labels, sample_size=SILHOUETTE_SAMPLE_SIZE):
    unique, counts = np.unique(labels, return_counts=True)
    if len(unique) < 2 or counts.min() < 2:
        return {"Silhouette": np.nan, "Calinski_Harabasz": np.nan, "Davies_Bouldin": np.nan}
    sil_kwargs = {}
    if sample_size is not None and len(X_space) > sample_size:
        sil_kwargs = {"sample_size": sample_size, "random_state": SEED}
    return {
        "Silhouette": silhouette_score(X_space, labels, **sil_kwargs),
        "Calinski_Harabasz": calinski_harabasz_score(X_space, labels),
        "Davies_Bouldin": davies_bouldin_score(X_space, labels),
    }

def fit_classical(X, seed=SEED):
    labels = {}
    labels["KMeans"] = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=seed).fit_predict(X)
    try:
        agg = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage="average", metric="euclidean")
    except TypeError:
        agg = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage="average", affinity="euclidean")
    labels["Agglomerative"] = agg.fit_predict(X)
    labels["GMM"] = GaussianMixture(
        n_components=N_CLUSTERS, covariance_type="full", max_iter=500,
        random_state=seed, init_params="kmeans", reg_covar=1e-4
    ).fit_predict(np.asarray(X, dtype=np.float64))
    return labels

scenario_arrays = {
    "legacy_median": X_legacy,
    "robust_median_mode": X_robust,
    "mice_1": mice_arrays[1],
}
all_labels = {}
metric_rows = []

for scenario, Xs in scenario_arrays.items():
    print("Méthodes classiques :", scenario, Xs.shape)
    fitted = fit_classical(Xs)
    for method, lab in fitted.items():
        all_labels[(scenario, method)] = lab
        row = {"scenario": scenario, "method": method, "space": "standardized_input"}
        row.update(clustering_metrics(Xs, lab))
        metric_rows.append(row)
        out = ids.copy()
        out["cluster"] = lab
        out.to_csv(OUT_DIR / f"labels_{scenario}_{method}.csv", index=False)

# Stabilité entre les cinq imputations MICE pour KMeans.
mice_kmeans_labels = {}
for draw, Xs in mice_arrays.items():
    lab = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=SEED).fit_predict(Xs)
    mice_kmeans_labels[draw] = lab

mice_stability_rows = []
for i in range(1, N_MICE_DRAWS + 1):
    for j in range(i + 1, N_MICE_DRAWS + 1):
        mice_stability_rows.append({"draw_i": i, "draw_j": j, "ARI": adjusted_rand_score(mice_kmeans_labels[i], mice_kmeans_labels[j])})
mice_stability = pd.DataFrame(mice_stability_rows)
mice_stability.to_csv(OUT_DIR / "mice_kmeans_pairwise_stability.csv", index=False)
display(mice_stability)


## 5. AE+KMeans, DEC et IDEC

Cette cellule utilise une architecture commune et un autoencodeur pré-entraîné commun par scénario. Elle évalue les labels dans l'espace latent optimisé et dans l'espace d'entrée standardisé. Pour un premier test rapide, mettez `RUN_DEEP_MODELS = False` dans la configuration ; remettez-le à `True` pour produire les résultats destinés à l'article.


In [ ]:
# 5. Modèles profonds
# Permet aussi d'exécuter cette cellule séparément après la préparation des matrices.
if "clustering_metrics" not in globals():
    def clustering_metrics(X_space, labels, sample_size=SILHOUETTE_SAMPLE_SIZE):
        unique, counts = np.unique(labels, return_counts=True)
        if len(unique) < 2 or counts.min() < 2:
            return {"Silhouette": np.nan, "Calinski_Harabasz": np.nan, "Davies_Bouldin": np.nan}
        sil_kwargs = {}
        if sample_size is not None and len(X_space) > sample_size:
            sil_kwargs = {"sample_size": sample_size, "random_state": SEED}
        return {
            "Silhouette": silhouette_score(X_space, labels, **sil_kwargs),
            "Calinski_Harabasz": calinski_harabasz_score(X_space, labels),
            "Davies_Bouldin": davies_bouldin_score(X_space, labels),
        }

if TORCH_AVAILABLE:
    class Autoencoder(nn.Module):
        def __init__(self, input_dim, hidden_dims=HIDDEN_DIMS, latent_dim=LATENT_DIM):
            super().__init__()
            enc = []
            prev = input_dim
            for h in hidden_dims:
                enc += [nn.Linear(prev, h), nn.ReLU()]
                prev = h
            enc += [nn.Linear(prev, latent_dim)]
            self.encoder = nn.Sequential(*enc)
            dec = []
            prev = latent_dim
            for h in reversed(hidden_dims):
                dec += [nn.Linear(prev, h), nn.ReLU()]
                prev = h
            dec += [nn.Linear(prev, input_dim)]
            self.decoder = nn.Sequential(*dec)

        def forward(self, x):
            z = self.encoder(x)
            return self.decoder(z), z

    def pretrain_autoencoder(X, seed=SEED):
        set_seed(seed)
        model = Autoencoder(X.shape[1]).to(DEVICE)
        tensor = torch.tensor(X, dtype=torch.float32)
        loader = DataLoader(TensorDataset(tensor), batch_size=BATCH_SIZE, shuffle=True)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        loss_fn = nn.MSELoss()
        model.train()
        for epoch in range(1, PRETRAIN_EPOCHS + 1):
            total = 0.0
            for (xb,) in loader:
                xb = xb.to(DEVICE)
                optimizer.zero_grad()
                recon, _ = model(xb)
                loss = loss_fn(recon, xb)
                loss.backward()
                optimizer.step()
                total += loss.item() * len(xb)
            if epoch == 1 or epoch % 10 == 0:
                print(f"AE epoch {epoch}/{PRETRAIN_EPOCHS}, reconstruction={total/len(X):.6f}")
        return model

    @torch.no_grad()
    def encode(model, X):
        model.eval()
        xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        return model.encoder(xt).cpu().numpy()

    def student_q(z, centers, alpha=1.0):
        dist = torch.sum((z.unsqueeze(1) - centers.unsqueeze(0)) ** 2, dim=2)
        q = (1.0 + dist / alpha) ** (-(alpha + 1.0) / 2.0)
        return q / torch.sum(q, dim=1, keepdim=True)

    def target_distribution(q):
        weight = (q ** 2) / torch.sum(q, dim=0, keepdim=True).clamp_min(1e-12)
        return weight / torch.sum(weight, dim=1, keepdim=True).clamp_min(1e-12)

    def refine_dec(pretrained, X, idec=False, seed=SEED):
        set_seed(seed)
        model = copy.deepcopy(pretrained).to(DEVICE)
        xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            z0 = model.encoder(xt).cpu().numpy()
        km = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=seed).fit(z0)
        centers = nn.Parameter(torch.tensor(km.cluster_centers_, dtype=torch.float32, device=DEVICE))
        optimizer = optim.Adam(list(model.parameters()) + [centers], lr=LEARNING_RATE)
        for step in range(1, DEEP_MAX_ITER + 1):
            model.train()
            optimizer.zero_grad()
            recon, z = model(xt)
            q = student_q(z, centers)
            p = target_distribution(q.detach())
            kl = torch.sum(p * torch.log((p + 1e-12) / (q + 1e-12))) / len(X)
            recon_loss = torch.mean((recon - xt) ** 2)
            loss = kl + IDEC_RECON_WEIGHT * recon_loss if idec else kl
            loss.backward()
            optimizer.step()
            if step % 200 == 0:
                print(f"{'IDEC' if idec else 'DEC'} step {step}/{DEEP_MAX_ITER}: total={loss.item():.6f}, KL={kl.item():.6f}")
        model.eval()
        with torch.no_grad():
            z = model.encoder(xt)
            q = student_q(z, centers)
            labels = torch.argmax(q, dim=1).cpu().numpy()
        return labels, z.cpu().numpy()

def run_deep_scenario(name, Xs):
    print("\n=== Modèles profonds :", name, Xs.shape, "===")
    pretrained = pretrain_autoencoder(Xs)
    Zae = encode(pretrained, Xs)
    ae_labels = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=SEED).fit_predict(Zae)
    dec_labels, Zdec = refine_dec(pretrained, Xs, idec=False)
    idec_labels, Zidec = refine_dec(pretrained, Xs, idec=True)
    return {
        "AE+KMeans": (ae_labels, Zae),
        "DEC": (dec_labels, Zdec),
        "IDEC": (idec_labels, Zidec),
    }

# Les représentations latentes sont conservées pour les analyses dans Z.
deep_latent_results = {}

if RUN_DEEP_MODELS:
    if not TORCH_AVAILABLE:
        warnings.warn("RUN_DEEP_MODELS=True mais PyTorch n'est pas installé.")
    else:
        for scenario in ["robust_median_mode", "mice_1"]:
            Xs = scenario_arrays[scenario]
            deep_results = run_deep_scenario(scenario, Xs)
            for method, (lab, Z) in deep_results.items():
                deep_latent_results[(scenario, method)] = (lab.copy(), Z.copy())
                all_labels[(scenario, method)] = lab
                latent_row = {"scenario": scenario, "method": method, "space": "latent"}
                latent_row.update(clustering_metrics(Z, lab))
                metric_rows.append(latent_row)
                input_row = {"scenario": scenario, "method": method, "space": "standardized_input"}
                input_row.update(clustering_metrics(Xs, lab))
                metric_rows.append(input_row)
                out = ids.copy()
                out["cluster"] = lab
                method_file = method.replace('+', '_').replace(' ', '_')
                out.to_csv(OUT_DIR / f"labels_{scenario}_{method_file}.csv", index=False)
                np.save(OUT_DIR / f"Z_{scenario}_{method_file}.npy", Z)

        # Profils scalaires calculés dans Z. Ils décrivent uniquement la géométrie
        # latente et ne doivent pas être interprétés comme des scores scolaires.
        latent_profile_rows = []
        latent_tsne_rows = []
        cluster_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
        display_names = {"AE+KMeans": "AE+k-means", "DEC": "DEC", "IDEC": "IDEC"}

        for scenario in ["robust_median_mode", "mice_1"]:
            available_methods = [m for m in ["AE+KMeans", "DEC", "IDEC"]
                                 if (scenario, m) in deep_latent_results]
            if not available_methods:
                continue

            fig, axes = plt.subplots(1, len(available_methods), figsize=(18, 5.2))
            axes = np.atleast_1d(axes)
            fig.patch.set_facecolor("white")

            for ax, method in zip(axes, available_methods):
                lab, Z = deep_latent_results[(scenario, method)]
                method_file = method.replace('+', '_').replace(' ', '_')

                for cluster in sorted(np.unique(lab)):
                    part = Z[lab == cluster]
                    latent_profile_rows.append({
                        "scenario": scenario,
                        "method": method,
                        "space": "Z",
                        "cluster": int(cluster) + 1,
                        "n": int(len(part)),
                        "signed_mean_Z": float(part.mean()),
                        "absolute_mean_Z": float(np.abs(part).mean()),
                    })

                # Chaque méthode possède son propre Z : le t-SNE est donc ajusté séparément.
                Z_tsne = TSNE(
                    n_components=2, perplexity=35, random_state=SEED,
                    init="pca", learning_rate="auto"
                ).fit_transform(Z)

                coords = ids.copy()
                coords["scenario"] = scenario
                coords["method"] = method
                coords["cluster"] = lab.astype(int) + 1
                coords["tsne_Z_1"] = Z_tsne[:, 0]
                coords["tsne_Z_2"] = Z_tsne[:, 1]
                latent_tsne_rows.append(coords)
                coords.to_csv(OUT_DIR / f"tsne_Z_{scenario}_{method_file}.csv", index=False)

                ax.set_facecolor("white")
                for cluster in sorted(np.unique(lab)):
                    mask = lab == cluster
                    ax.scatter(
                        Z_tsne[mask, 0], Z_tsne[mask, 1], s=9,
                        color=cluster_colors[int(cluster) % len(cluster_colors)],
                        alpha=0.82, linewidths=0, label=f"C{int(cluster) + 1}"
                    )
                ax.set_title(f"{display_names[method]}: t-SNE in Z", fontweight="bold")
                ax.set_xlabel("t-SNE Z1")
                ax.set_ylabel("t-SNE Z2")
                ax.legend(title="Cluster", fontsize=7, title_fontsize=8, markerscale=1.5)

            fig.suptitle(f"Deep-method latent representations ({scenario})", fontsize=14)
            fig.tight_layout(rect=[0, 0, 1, 0.95], pad=2.0)
            fig.savefig(
                OUT_DIR / f"tsne_Z_deep_methods_{scenario}.png",
                dpi=300, bbox_inches="tight", facecolor="white"
            )
            display(fig)
            plt.close(fig)

        latent_profiles_df = pd.DataFrame(latent_profile_rows)
        latent_profiles_df.to_csv(OUT_DIR / "latent_signed_absolute_means_Z.csv", index=False)
        if latent_tsne_rows:
            pd.concat(latent_tsne_rows, ignore_index=True).to_csv(
                OUT_DIR / "latent_tsne_coordinates_Z.csv", index=False
            )
        display(latent_profiles_df)

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(OUT_DIR / "all_clustering_metrics.csv", index=False)
display(metrics_df)


## 6. Comparaison finale : ancien traitement, scénario robuste et MICE

Les tableaux produits ici sont ceux à consulter avant de modifier l'article. Une faible valeur d'ARI signifie que les profils dépendent fortement du prétraitement.


In [ ]:
# 6. ARI entre scénarios recalculés
ari_rows = []
methods = sorted({method for _, method in all_labels})
for method in methods:
    comparisons = [
        ("legacy_median", "robust_median_mode"),
        ("robust_median_mode", "mice_1"),
        ("legacy_median", "mice_1"),
    ]
    for a, b in comparisons:
        if (a, method) in all_labels and (b, method) in all_labels:
            ari_rows.append({
                "method": method,
                "scenario_a": a,
                "scenario_b": b,
                "ARI": adjusted_rand_score(all_labels[(a, method)], all_labels[(b, method)]),
            })
ari_df = pd.DataFrame(ari_rows)
ari_df.to_csv(OUT_DIR / "ari_between_preprocessing_scenarios.csv", index=False)
display(ari_df)

# Comparaison facultative avec les labels déjà enregistrés par l'ancien notebook.
OLD_LABEL_FILES = {
    "KMeans": DATA_DIR / "KMEANS_results_MAROC" / "ids_with_clusters_kmeans.csv",
    "Agglomerative": DATA_DIR / "AGGLO_results_MAROC_average" / "ids_with_clusters_agglo.csv",
    "GMM": DATA_DIR / "GMM_results_MAROC" / "ids_with_clusters_gmm.csv",
    "AE+KMeans": DATA_DIR / "AE_KMEANS_results_MAROC" / "ids_with_clusters_AE_kmeans.csv",
    "DEC": DATA_DIR / "DEC_results_MAROC_multi_seeds" / "seed_42" / "ids_with_clusters.csv",
    "IDEC": DATA_DIR / "IDEC_results_MAROC_multi_seeds" / "seed_42" / "ids_with_clusters.csv",
}

old_comparison_rows = []
for method, path in OLD_LABEL_FILES.items():
    if not path.exists() or ("robust_median_mode", method) not in all_labels:
        continue
    old = pd.read_csv(path)[["IDSTUD", "cluster"]].rename(columns={"cluster": "old_cluster"})
    new = ids[["IDSTUD"]].copy()
    new["new_cluster"] = all_labels[("robust_median_mode", method)]
    joined = old.merge(new, on="IDSTUD", how="inner")
    old_comparison_rows.append({
        "method": method,
        "n_matched": len(joined),
        "ARI_old_saved_vs_robust": adjusted_rand_score(joined["old_cluster"], joined["new_cluster"]),
        "old_file": str(path),
    })
old_comparison = pd.DataFrame(old_comparison_rows)
old_comparison.to_csv(OUT_DIR / "ari_old_saved_vs_robust.csv", index=False)
display(old_comparison)

# Résumé directement exploitable pour décider si les tableaux de l'article doivent être remplacés.
summary_lines = [
    "RÈGLE D'INTERPRÉTATION (indicative, à commenter avec les autres résultats)",
    "ARI >= 0.80 : partitions très proches ; 0.50-0.80 : différences modérées ; ARI < 0.50 : forte sensibilité.",
    "",
    "ARI entre scénarios :",
    ari_df.to_string(index=False) if not ari_df.empty else "Aucun résultat.",
    "",
    "ARI anciens fichiers vs scénario robuste :",
    old_comparison.to_string(index=False) if not old_comparison.empty else "Aucun résultat.",
    "",
    "Stabilité KMeans entre imputations MICE :",
    mice_stability.describe().to_string() if not mice_stability.empty else "Aucun résultat.",
]
(OUT_DIR / "comparison_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")
print("Tous les résultats ont été enregistrés dans :", OUT_DIR.resolve())


## 7. Comment utiliser les résultats dans l'article

1. Vérifier `source_variable_audit.csv` et `final_missingness_audit.csv`.
2. Lire `ari_between_preprocessing_scenarios.csv` et `ari_old_saved_vs_robust.csv`.
3. Si les ARI sont élevés, présenter le nouveau pipeline comme une analyse de robustesse confirmant les conclusions.
4. Si les ARI sont faibles, remplacer les anciens tableaux/figures par les résultats robustes et reconnaître la sensibilité au prétraitement.
5. Ne jamais présenter `IterativeImputer` comme un pooling complet au sens des règles d'inférence de Rubin : ici, les cinq jeux servent à évaluer la stabilité des partitions.


## 8. Sorties des analyses dans l'espace latent Z

- Les matrices `Z_<scenario>_<method>.npy` conservent les representations latentes des trois methodes profondes.
- `latent_signed_absolute_means_Z.csv` contient les moyennes signee et absolue calculees dans Z par cluster. Elles decrivent la geometrie latente, et non la performance scolaire.
- `tsne_Z_deep_methods_<scenario>.png` contient les trois projections t-SNE latentes.
- `latent_tsne_coordinates_Z.csv` et les fichiers `tsne_Z_<scenario>_<method>.csv` contiennent les coordonnees correspondantes.
- Ces sorties sont des diagnostics de la representation apprise et ne constituent pas une validation externe.
